# Student-1 — Error Progression: Lesson 2 → Lesson 3

Two levels of comparison:
- **Fine-grained** — exact `rule_id` match (same mistake)
- **Coarse** — same `grammar_category` (similar mistake type)

Three columns: **Resolved** (fixed ✓) · **Persisted** (still struggling) · **New** (appeared in L3)

In [1]:
import sys, json, pathlib
from collections import Counter
from IPython.display import display, HTML

here = pathlib.Path().resolve()
root = here
for _ in range(4):
    if (root / 'grammar_errors').is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

ERRORS_DIR = root / 'processed_data' / 'errors'   / 'Student-1'
SENT_DIR   = root / 'processed_data' / 'sentences' / 'Student-1'

def load_errors(lesson):
    return json.loads((ERRORS_DIR / lesson / 'grammar_errors.json').read_text(encoding='utf-8'))

def count_sentences(lesson):
    txt = (SENT_DIR / lesson / 'validated-sentence-separation.txt').read_text(encoding='utf-8')
    return sum(1 for l in txt.splitlines() if l.strip())

e2, e3 = load_errors('lesson-2'), load_errors('lesson-3')
n2, n3 = count_sentences('lesson-2'), count_sentences('lesson-3')

print('Lesson-2:', len(e2), 'errors in', n2, 'sentences')
print('Lesson-3:', len(e3), 'errors in', n3, 'sentences')

Lesson-2: 19 errors in 187 sentences
Lesson-3: 35 errors in 303 sentences


In [2]:
# ── Colour palette ──────────────────────────────────────────────────────────
DIM = {
    'A': {'col': '#3B82F6', 'bg': '#DBEAFE', 'dark': '#1D4ED8', 'label': 'Sentence Architecture'},
    'B': {'col': '#F59E0B', 'bg': '#FEF3C7', 'dark': '#92400E', 'label': 'Tense & Aspect Mastery'},
    'C': {'col': '#10B981', 'bg': '#D1FAE5', 'dark': '#065F46', 'label': 'Nominal Precision'},
    'D': {'col': '#8B5CF6', 'bg': '#EDE9FE', 'dark': '#4C1D95', 'label': 'Modal & Functional Range'},
}

def badge(dim, text=None):
    t = text or dim
    st = 'background:' + DIM[dim]['dark'] + ';color:white;padding:1px 7px;border-radius:8px;font-size:11px;font-weight:700;'
    return '<span style="' + st + '">' + t + '</span>'

def tag(el, content, style='', extra=''):
    """Build <el style="...">content</el>."""
    s = ' style="' + style + '"' if style else ''
    return '<' + el + s + extra + '>' + content + '</' + el + '>'

# ── Overview card: total counts + error rate ────────────────────────────────
rate2 = round(100 * len(e2) / n2, 1)
rate3 = round(100 * len(e3) / n3, 1)
delta = rate3 - rate2
trend_color = '#16A34A' if delta < -0.5 else '#DC2626' if delta > 0.5 else '#D97706'
trend_text  = ('DOWN  improved' if delta < -0.5
               else 'UP  increased' if delta > 0.5
               else 'STABLE')
sign = '+' if delta >= 0 else ''

def lesson_box(errors, n_sents, label):
    rate = round(100 * len(errors) / n_sents, 1)
    inner = (tag('div', label,       'font-size:13px;font-weight:600;color:#6B7280;margin-bottom:4px;')
           + tag('div', str(len(errors)), 'font-size:36px;font-weight:800;color:#111827;')
           + tag('div', str(n_sents) + ' sentences &nbsp;&middot;&nbsp; ' + str(rate) + '% error rate',
                 'font-size:12px;color:#6B7280;'))
    return tag('div', inner, 'flex:1;text-align:center;padding:16px 20px;border-radius:8px;background:#F9FAFB;')

arrow_inner = tag('div',
    trend_text + tag('span', sign + str(round(delta, 1)) + ' pp', 'font-size:12px;display:block;'),
    'font-size:18px;font-weight:700;color:' + trend_color + ';text-align:center;')
arrow_box = tag('div', arrow_inner, 'padding:0 20px;')

overview_inner = (tag('h3', 'Overall error rate', 'margin:0 0 14px;font-size:15px;color:#374151;')
                + tag('div', lesson_box(e2, n2, 'Lesson 2') + arrow_box + lesson_box(e3, n3, 'Lesson 3'),
                       'display:flex;gap:12px;align-items:center;'))
overview = tag('div', overview_inner,
               'font-family:sans-serif;border:1px solid #E5E7EB;border-radius:10px;padding:16px;margin-bottom:20px;')

# ── Dimension side-by-side bars ──────────────────────────────────────────────
dim_c2 = Counter(e['dimension_code'] for e in e2)
dim_c3 = Counter(e['dimension_code'] for e in e3)
max_val = max(max(dim_c2.values(), default=1), max(dim_c3.values(), default=1))

dim_rows = ''
for dim in ['A', 'B', 'C', 'D']:
    c2v = dim_c2.get(dim, 0)
    c3v = dim_c3.get(dim, 0)
    w2  = round(100 * c2v / max_val)
    w3  = round(100 * c3v / max_val)
    col = DIM[dim]['col']
    bar2 = tag('div', '', 'height:14px;width:' + str(w2) + '%;background:' + col + '99;border-radius:2px;min-width:2px;')
    bar3 = tag('div', '', 'height:14px;width:' + str(w3) + '%;background:' + col + ';border-radius:2px;min-width:2px;')
    chg  = c3v - c2v
    chg_s = ('+' if chg > 0 else '') + str(chg)
    chg_c = '#DC2626' if chg > 0 else '#16A34A' if chg < 0 else '#6B7280'
    cell2 = tag('div', bar2 + tag('span', str(c2v), 'font-size:11px;color:#6B7280;margin-left:4px;'), 'display:flex;align-items:center;')
    cell3 = tag('div', bar3 + tag('span', str(c3v), 'font-size:11px;color:#6B7280;margin-left:4px;'), 'display:flex;align-items:center;')
    dim_rows += ('<tr style="vertical-align:middle;">'
                 + tag('td', badge(dim), 'padding:5px 10px;width:24px;')
                 + tag('td', DIM[dim]['label'], 'padding:5px 10px;font-size:11px;color:#374151;width:180px;')
                 + tag('td', cell2, 'padding:5px 10px;width:90px;')
                 + tag('td', cell3, 'padding:5px 10px;width:90px;')
                 + tag('td', chg_s, 'padding:5px 10px;font-weight:700;font-size:12px;color:' + chg_c + ';')
                 + '</tr>')

thead = ('<tr style="font-size:11px;font-weight:700;color:#9CA3AF;text-transform:uppercase;">'
         + '<th></th>'
         + tag('th', 'Dimension', 'padding:4px 10px;')
         + tag('th', 'Lesson 2',  'padding:4px 10px;')
         + tag('th', 'Lesson 3',  'padding:4px 10px;')
         + tag('th', 'Change',    'padding:4px 10px;')
         + '</tr>')
dim_table = tag('table', thead + dim_rows, 'border-collapse:collapse;width:100%;')
dim_card = tag('div',
    tag('h3', 'Errors by Proficiency Dimension', 'margin:0 0 14px;font-size:15px;color:#374151;') + dim_table,
    'font-family:sans-serif;border:1px solid #E5E7EB;border-radius:10px;padding:16px;margin-bottom:20px;')

display(HTML(overview + dim_card))

,Dimension,Lesson 2,Lesson 3,Change
A,Sentence Architecture,3,3,0
B,Tense & Aspect Mastery,12,19,+7
C,Nominal Precision,3,11,+8
D,Modal & Functional Range,1,2,+1


In [3]:
# ── FINE-GRAINED: exact rule_id ─────────────────────────────────────────────

from collections import defaultdict

cnt2 = Counter((e['rule_id'], e['grammar_category'], e['dimension_code']) for e in e2)
cnt3 = Counter((e['rule_id'], e['grammar_category'], e['dimension_code']) for e in e3)
all_keys = set(cnt2) | set(cnt3)

resolved  = sorted([(k, cnt2[k])          for k in all_keys if k in cnt2 and k not in cnt3], key=lambda x: -x[1])
persisted = sorted([(k, cnt2[k], cnt3[k]) for k in all_keys if k in cnt2 and k in cnt3],     key=lambda x: -(x[1]+x[2]))
new       = sorted([(k, cnt3[k])          for k in all_keys if k not in cnt2 and k in cnt3], key=lambda x: -x[1])

# Example lookup: rule_id -> [error_record, ...]
ex2 = defaultdict(list)
ex3 = defaultdict(list)
for e in e2: ex2[e['rule_id']].append(e)
for e in e3: ex3[e['rule_id']].append(e)

def highlight(sentence, offset, length, dim):
    col    = DIM[dim]['col']
    before = sentence[:offset]
    match  = sentence[offset:offset + length]
    after  = sentence[offset + length:]
    mst    = 'background:' + col + '25;border-bottom:2px solid ' + col + ';padding:1px 0;font-weight:700;'
    return before + tag('mark', match, mst) + after

def example_lines(records, dim, label=None, max_ex=2):
    lines = []
    seen  = set()
    for rec in records:
        key = rec['matched_text'].lower()
        if key in seen:
            continue
        seen.add(key)
        sent_hl = highlight(rec['sentence'], rec['offset'], rec['error_length'], dim)
        prefix  = (tag('span', label + '&nbsp;', 'color:#9CA3AF;font-size:10px;font-weight:700;font-style:normal;')
                   if label else '')
        lines.append(prefix + tag('span', sent_hl,
                     'font-size:11px;font-style:italic;color:#4B5563;font-family:Georgia,serif;line-height:1.6;'))
        if len(lines) >= max_ex:
            break
    return lines

def example_row(lines, ncols=4):
    if not lines:
        return ''
    bar  = tag('div', '', 'width:2px;min-width:2px;background:#E5E7EB;margin-right:10px;border-radius:1px;')
    body = tag('div', '<br>'.join(lines), 'flex:1;min-width:0;')
    inner = tag('div', bar + body, 'display:flex;padding:4px 8px 10px 28px;')
    return ('<tr>' + tag('td', inner, 'padding:0;background:#FAFAFA;',
                          ' colspan="' + str(ncols) + '"') + '</tr>')

def count_cell(n2=None, n3=None, mode='resolved'):
    if mode == 'resolved':
        return tag('td', 'L2:&nbsp;' + tag('b', str(n2)),
                   'padding:6px 8px;font-size:12px;color:#6B7280;text-align:right;white-space:nowrap;')
    if mode == 'new':
        return tag('td', 'L3:&nbsp;' + tag('b', str(n3)),
                   'padding:6px 8px;font-size:12px;color:#6B7280;text-align:right;white-space:nowrap;')
    chg   = n3 - n2
    chg_s = ('+' if chg > 0 else '') + str(chg)
    chg_c = '#DC2626' if chg > 0 else '#16A34A' if chg < 0 else '#6B7280'
    txt   = (str(n2) + tag('span', ' &rarr; ' + str(n3) + '&nbsp;'
             + tag('b', chg_s, 'color:' + chg_c + ';'), 'color:#6B7280;'))
    return tag('td', txt, 'padding:6px 8px;font-size:12px;text-align:right;white-space:nowrap;')

def rule_row(key, n2=None, n3=None, mode='resolved'):
    rid, cat, dim = key
    main = ('<tr style="border-top:1px solid #F0F0F0;">'
            + tag('td', badge(dim),  'padding:6px 8px;vertical-align:top;')
            + tag('td', cat,         'padding:6px 8px;font-size:12px;vertical-align:top;')
            + tag('td', rid,         'padding:6px 8px;font-size:11px;color:#6B7280;font-family:monospace;vertical-align:top;')
            + count_cell(n2=n2, n3=n3, mode=mode)
            + '</tr>')
    if mode == 'resolved':
        lines = example_lines(ex2[rid], dim, max_ex=2)
    elif mode == 'new':
        lines = example_lines(ex3[rid], dim, max_ex=2)
    else:  # persisted: one from each lesson to see if the error looks similar
        lines = (example_lines(ex2[rid], dim, label='L2', max_ex=1)
               + example_lines(ex3[rid], dim, label='L3', max_ex=1))
    return main + example_row(lines)

def panel(title, color, icon, rows, summary):
    thead = ('<tr style="font-size:10px;font-weight:700;color:#9CA3AF;text-transform:uppercase;background:#F9FAFB;">'
             + tag('th', 'Dim',      'padding:4px 8px;')
             + tag('th', 'Category', 'padding:4px 8px;')
             + tag('th', 'Rule ID',  'padding:4px 8px;')
             + tag('th', 'Count',    'padding:4px 8px;text-align:right;')
             + '</tr>')
    table  = tag('table', thead + rows, 'border-collapse:collapse;width:100%;font-family:sans-serif;')
    scroll = tag('div', table, 'overflow-y:auto;max-height:520px;')
    bst    = 'float:right;background:rgba(255,255,255,0.3);color:white;border-radius:10px;padding:1px 8px;font-size:12px;font-weight:700;'
    header = tag('div',
                 icon + '&nbsp;' + tag('span', title, 'font-weight:700;color:white;font-size:14px;')
                 + tag('span', summary, bst),
                 'background:' + color + ';padding:10px 14px;')
    return tag('div', header + scroll,
               'flex:1;border:2px solid ' + color + ';border-radius:10px;overflow:hidden;min-width:0;')

res_rows = ''.join(rule_row(k, n2=n,       mode='resolved')  for k, n    in resolved)
per_rows = ''.join(rule_row(k, n2=a, n3=b, mode='persisted') for k, a, b in persisted)
new_rows = ''.join(rule_row(k,       n3=n, mode='new')       for k, n    in new)

three = tag('div',
    panel('Resolved',  '#16A34A', '&#10003;', res_rows,
          str(len(resolved))  + ' types &middot; ' + str(sum(n for _,n in resolved))  + ' errors fixed')
  + panel('Persisted', '#D97706', '&#126;',   per_rows,
          str(len(persisted)) + ' types still present')
  + panel('New',       '#DC2626', '&#10007;', new_rows,
          str(len(new))       + ' types &middot; ' + str(sum(n for _,n in new))       + ' new errors'),
    'display:flex;gap:14px;font-family:sans-serif;')

display(HTML(
    tag('h3', 'Fine-grained &mdash; exact rule ID', 'font-family:sans-serif;margin-bottom:10px;')
    + three
))

In [4]:
# ── COARSE: grammar_category ────────────────────────────────────────────────

cat2 = Counter((e['grammar_category'], e['dimension_code']) for e in e2)
cat3 = Counter((e['grammar_category'], e['dimension_code']) for e in e3)
cat_all = set(cat2) | set(cat3)

cat_res  = sorted([(k, cat2[k])          for k in cat_all if k in cat2 and k not in cat3], key=lambda x: -x[1])
cat_per  = sorted([(k, cat2[k], cat3[k]) for k in cat_all if k in cat2 and k in cat3],     key=lambda x: -(x[1]+x[2]))
cat_new  = sorted([(k, cat3[k])          for k in cat_all if k not in cat2 and k in cat3], key=lambda x: -x[1])

# Example lookup by (grammar_category, dim)
cex2 = defaultdict(list)
cex3 = defaultdict(list)
for e in e2: cex2[(e['grammar_category'], e['dimension_code'])].append(e)
for e in e3: cex3[(e['grammar_category'], e['dimension_code'])].append(e)

def cat_row(key, n2=None, n3=None, mode='resolved'):
    cat, dim = key
    main = ('<tr style="border-top:1px solid #F0F0F0;">'
            + tag('td', badge(dim), 'padding:6px 8px;vertical-align:top;')
            + tag('td', cat,        'padding:6px 8px;font-size:13px;vertical-align:top;')
            + count_cell(n2=n2, n3=n3, mode=mode)
            + '</tr>')
    if mode == 'resolved':
        lines = example_lines(cex2[key], dim, max_ex=2)
    elif mode == 'new':
        lines = example_lines(cex3[key], dim, max_ex=2)
    else:
        lines = (example_lines(cex2[key], dim, label='L2', max_ex=1)
               + example_lines(cex3[key], dim, label='L3', max_ex=1))
    return main + example_row(lines, ncols=3)

def cat_panel(title, color, icon, rows, summary):
    thead = ('<tr style="font-size:10px;font-weight:700;color:#9CA3AF;text-transform:uppercase;background:#F9FAFB;">'
             + tag('th', 'Dim',      'padding:4px 8px;')
             + tag('th', 'Category', 'padding:4px 8px;')
             + tag('th', 'Count',    'padding:4px 8px;text-align:right;')
             + '</tr>')
    table  = tag('table', thead + rows, 'border-collapse:collapse;width:100%;font-family:sans-serif;')
    scroll = tag('div', table, 'overflow-y:auto;max-height:420px;')
    bst    = 'float:right;background:rgba(255,255,255,0.3);color:white;border-radius:10px;padding:1px 8px;font-size:12px;font-weight:700;'
    header = tag('div',
                 icon + '&nbsp;' + tag('span', title, 'font-weight:700;color:white;font-size:14px;')
                 + tag('span', summary, bst),
                 'background:' + color + ';padding:10px 14px;')
    return tag('div', header + scroll,
               'flex:1;border:2px solid ' + color + ';border-radius:10px;overflow:hidden;min-width:0;')

cr_rows = ''.join(cat_row(k, n2=n,       mode='resolved')  for k, n    in cat_res)
cp_rows = ''.join(cat_row(k, n2=a, n3=b, mode='persisted') for k, a, b in cat_per)
cn_rows = ''.join(cat_row(k,       n3=n, mode='new')       for k, n    in cat_new)

cat_three = tag('div',
    cat_panel('Resolved',  '#16A34A', '&#10003;', cr_rows, str(len(cat_res)) + ' categories')
  + cat_panel('Persisted', '#D97706', '&#126;',   cp_rows, str(len(cat_per)) + ' categories')
  + cat_panel('New',       '#DC2626', '&#10007;', cn_rows, str(len(cat_new)) + ' categories'),
    'display:flex;gap:14px;font-family:sans-serif;')

display(HTML(
    tag('h3', 'Coarse &mdash; grammar category', 'font-family:sans-serif;margin-bottom:10px;margin-top:28px;')
    + cat_three
))